In [ ]:
from pathlib import Path
import pandas as pd
import h5py
import scipy.io
import numpy as np

mat_path = Path("../data/EEG_DDvMMR_photodiode_test_20260622.mat")
if not mat_path.exists():  # Supports kernels started from the repository root.
    mat_path = Path("data/EEG_DDvMMR_photodiode_test_20260622.mat")
print(f"Loading MAT file: {mat_path}")

try:
    mat = scipy.io.loadmat(mat_path, squeeze_me=True, struct_as_record=False)
except NotImplementedError:
    # MATLAB v7.3 files use HDF5, which scipy.io.loadmat cannot read.
    with h5py.File(mat_path, "r") as h5_file:
        mat = {key: h5_file[key][()] for key in h5_file.keys()}
    print("Loaded MATLAB v7.3/HDF5 file with h5py.")

keys = [k for k in mat.keys() if not k.startswith("__")]
print("MAT variables:", keys)
print(f"Number of variables loaded: {len(keys)}")

frames = {}
for key in keys:
    value = mat[key]
    if isinstance(value, np.ndarray):
        if value.ndim == 2:
            frames[key] = pd.DataFrame(value)
        elif value.ndim == 1:
            frames[key] = pd.DataFrame(value, columns=[key])
        else:
            print(f"Skipping {key}: ndarray with ndim={value.ndim}")
    elif np.isscalar(value) or isinstance(value, (str, bytes)):
        frames[key] = pd.DataFrame({key: [value]})
    else:
        print(f"Skipping {key}: unsupported type {type(value)}")

print("\nImported DataFrames:")
for key, df in frames.items():
    print(f"- {key}: shape={df.shape}, columns={list(df.columns)[:10]}")
    print(df.head())
    print()

if not frames:
    print("No suitable variables were converted to DataFrames.")

In [ ]:
import h5py
import numpy as np
import pandas as pd
from pathlib import Path

mat_path = Path("../data/EEG_DDvMMR_photodiode_test_20260622.mat")
if not mat_path.exists():  # Supports kernels started from the repository root.
    mat_path = Path("data/EEG_DDvMMR_photodiode_test_20260622.mat")

with h5py.File(mat_path, "r") as f:
    eeg_array = f["EEG_data"][()]  # shape=(26881, 67), float64

df_eeg = pd.DataFrame(eeg_array)
df_eeg.columns = [f"ch_{i}" for i in range(df_eeg.shape[1])]

df_eeg.info()